In [1]:
import pandas as pd
df = pd.read_csv('pitchbook_combined_processed_results.csv')

In [2]:
# Compare o4-mini LLM baseline to sample-based baselines
# Matches the logic from compare_models.py compute_llm_prior_win_rates_over_n_sample_baselines
# LLM is "better" when it has lower error than the baseline

llm_approach = 'o4-mini_base_direct_tempmedium'
baseline_approaches = ['statistical_baseline_n5', 'statistical_baseline_n10',
                       'statistical_baseline_n20', 'statistical_baseline_n30']

# Map baseline approach to sample size
sample_size_map = {
    'statistical_baseline_n5': '5',
    'statistical_baseline_n10': '10',
    'statistical_baseline_n20': '20',
    'statistical_baseline_n30': '30'
}

# Error metrics to compare
error_metrics = ['abs_error_from_mean']

# Get unique variables
variables = df['variable'].unique()

print("="*80)
print("o4-mini PRIOR vs Sample-Based Baseline Comparison")
print("(Matching fitted distribution types with fallback)")
print("="*80)
print()

for metric in error_metrics:
    print(f"\n{'='*80}")
    print(f"Metric: {metric}")
    print(f"{'='*80}\n")
    
    results = {}
    
    for variable in variables:
        var_df = df[df['variable'] == variable].copy()
        
        print(f"\n{variable}:")
        print("-" * 60)
        
        # Get LLM baseline error and fitted distribution type
        llm_df = var_df[var_df['approach'] == llm_approach]
        
        if len(llm_df) == 0:
            print("  No o4-mini data found")
            continue
        
        llm_avg_error = llm_df[metric].mean()
        
        # Determine the fitted distribution type for this variable
        fitted_dist_types = llm_df['fitted_distribution_type'].dropna().unique()
        is_lognormal = len(fitted_dist_types) > 0 and fitted_dist_types[0] == 'lognormal'
        llm_fitted_dist = fitted_dist_types[0] if len(fitted_dist_types) > 0 else 'unknown'
        
        print(f"  o4-mini average {metric}: {llm_avg_error:.4f} (fitted: {llm_fitted_dist})")
        
        results[variable] = {}
        
        # Compare with each baseline - matching the distribution type with fallback
        for baseline in baseline_approaches:
            sample_size = sample_size_map[baseline]
            
            # Select the appropriate statistical baseline for this sample size
            stat_baseline = var_df[
                (var_df['approach'].str.contains('statistical')) &
                (var_df['sample_size'] == sample_size)
            ]
            
            # Filter by distribution type (with fallback logic from compare_models.py)
            if is_lognormal:
                stat_baseline_typed = stat_baseline[stat_baseline['fitted_distribution_type'] == 'lognormal']
                if not stat_baseline_typed.empty:
                    stat_baseline = stat_baseline_typed
            else:
                stat_baseline_typed = stat_baseline[stat_baseline['fitted_distribution_type'] != 'lognormal']
                if not stat_baseline_typed.empty:
                    stat_baseline = stat_baseline_typed
            
            if len(stat_baseline) == 0:
                print(f"  {baseline}: No data")
                continue
            
            baseline_avg_error = stat_baseline[metric].mean()
            
            # LLM is better when its error is lower
            llm_better = llm_avg_error < baseline_avg_error
            
            results[variable][baseline] = {
                'llm_avg': llm_avg_error,
                'baseline_avg': baseline_avg_error,
                'llm_better': llm_better,
                'fitted_dist': llm_fitted_dist
            }
            
            better_str = "✓ o4-mini BETTER" if llm_better else "✗ Baseline better"
            print(f"  {baseline}: {baseline_avg_error:.4f} - {better_str}")
    
    # Calculate overall percentages for this metric
    print(f"\n{'='*80}")
    print(f"SUMMARY for {metric}")
    print(f"{'='*80}")
    
    for baseline in baseline_approaches:
        total_vars = 0
        llm_better_count = 0
        
        for variable in results:
            if baseline in results[variable]:
                total_vars += 1
                if results[variable][baseline]['llm_better']:
                    llm_better_count += 1
        
        if total_vars > 0:
            percentage = (llm_better_count / total_vars) * 100
            print(f"{baseline}: o4-mini better {llm_better_count}/{total_vars} times ({percentage:.1f}%)")

print("\n" + "="*80)
print("Analysis complete!")
print("="*80)

o4-mini PRIOR vs Sample-Based Baseline Comparison
(Matching fitted distribution types with fallback)


Metric: abs_error_from_mean


Employees:
------------------------------------------------------------
  o4-mini average abs_error_from_mean: 24.9446 (fitted: lognormal)
  statistical_baseline_n5: 29.0317 - ✓ o4-mini BETTER
  statistical_baseline_n10: 28.9531 - ✓ o4-mini BETTER
  statistical_baseline_n20: 32.2781 - ✓ o4-mini BETTER
  statistical_baseline_n30: 30.4036 - ✓ o4-mini BETTER

IsTechCompany:
------------------------------------------------------------
  o4-mini average abs_error_from_mean: 0.2516 (fitted: beta)
  statistical_baseline_n5: 0.1226 - ✗ Baseline better
  statistical_baseline_n10: 0.1041 - ✗ Baseline better
  statistical_baseline_n20: 0.0783 - ✗ Baseline better
  statistical_baseline_n30: 0.0672 - ✗ Baseline better

IsUSBased:
------------------------------------------------------------
  o4-mini average abs_error_from_mean: 0.2363 (fitted: beta)
  statistical_base

In [3]:
# Load posteriors data
import numpy as np
from scipy import stats
import warnings

posteriors_df = pd.read_csv('pitchbook_combined_processed_results_with_posteriors.csv', low_memory=False)

# Get only o4-mini posteriors
o4_posteriors = posteriors_df[
    posteriors_df['approach'].str.contains('posterior') &
    posteriors_df['approach'].str.contains('o4-mini')
].copy()

print(f"Loaded {len(o4_posteriors)} o4-mini posterior rows")
print(f"Unique approaches: {o4_posteriors['approach'].unique()}")
print(f"Sample sizes: {sorted(o4_posteriors['sample_size'].dropna().unique())}")

Loaded 6161 o4-mini posterior rows
Unique approaches: ['o4-mini_base_direct_tempmedium_posterior_N10'
 'o4-mini_base_direct_tempmedium_posterior_N20'
 'o4-mini_base_direct_tempmedium_posterior_N30'
 'o4-mini_base_direct_tempmedium_posterior_N5'
 'o4-mini_base_direct_tempmedium_posterior_NALL']
Sample sizes: ['10', '20', '30', '5', 'ALL']


In [4]:
# Preprocess posteriors to compute mean, std, and error metrics
# This matches the logic from compare_models.py preprocess_posteriors function

def compute_beta_mean_median_mode(a, b):
    mean = stats.beta.mean(a, b)
    median = stats.beta.median(a, b)
    if a > 1 and b > 1:
        mode = (a - 1) / (a + b - 2)
    else:
        mode = np.nan
    return mean, median, mode

def compute_lognormal_mean_median_mode(mu, sigma):
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings('error')
            mean = np.exp(mu + sigma**2 / 2)
            median = np.exp(mu)
            mode = np.exp(mu - sigma**2)
    except (RuntimeWarning, OverflowError):
        mean = np.nan
        median = np.nan
        mode = np.nan
    return mean, median, mode

def compute_normal_mean_median_mode(mu, sigma):
    return mu, mu, mu

def compute_beta_std(a, b):
    return stats.beta.std(a, b)

def compute_lognormal_std(mu, sigma):
    variance = (np.exp(sigma**2) - 1) * np.exp(2*mu + sigma**2)
    return np.sqrt(variance)

def compute_normal_std(mu, sigma):
    return sigma

# Identify distribution types
beta_vars = o4_posteriors['fitted_distribution_type'] == 'beta'
lognorm_vars = o4_posteriors['fitted_distribution_type'] == 'lognormal'
norm_vars = o4_posteriors['fitted_distribution_type'] == 'gaussian'

# Compute mean, median, mode
if beta_vars.any():
    beta_results = o4_posteriors.loc[beta_vars].apply(
        lambda row: pd.Series(compute_beta_mean_median_mode(row['a'], row['b'])), axis=1).values
    o4_posteriors.loc[beta_vars, ['mean', 'median', 'mode']] = beta_results

if lognorm_vars.any():
    lognorm_results = o4_posteriors.loc[lognorm_vars].apply(
        lambda row: pd.Series(compute_lognormal_mean_median_mode(row['mu'], row['sigma'])), axis=1).values
    o4_posteriors.loc[lognorm_vars, ['mean', 'median', 'mode']] = lognorm_results

if norm_vars.any():
    norm_results = o4_posteriors.loc[norm_vars].apply(
        lambda row: pd.Series(compute_normal_mean_median_mode(row['mu'], row['sigma'])), axis=1).values
    o4_posteriors.loc[norm_vars, ['mean', 'median', 'mode']] = norm_results

# Compute std
if beta_vars.any():
    o4_posteriors.loc[beta_vars, 'std'] = o4_posteriors.loc[beta_vars].apply(
        lambda row: compute_beta_std(row['a'], row['b']), axis=1).values

if lognorm_vars.any():
    o4_posteriors.loc[lognorm_vars, 'std'] = o4_posteriors.loc[lognorm_vars].apply(
        lambda row: compute_lognormal_std(row['mu'], row['sigma']), axis=1).values

if norm_vars.any():
    o4_posteriors.loc[norm_vars, 'std'] = o4_posteriors.loc[norm_vars].apply(
        lambda row: compute_normal_std(row['mu'], row['sigma']), axis=1).values

# Compute absolute errors
o4_posteriors.loc[:, 'abs_error_from_mean'] = np.abs(o4_posteriors['ground_truth'] - o4_posteriors['mean'])
o4_posteriors.loc[:, 'abs_error_from_median'] = np.abs(o4_posteriors['ground_truth'] - o4_posteriors['median'])
o4_posteriors.loc[:, 'abs_error_from_mode'] = np.abs(o4_posteriors['ground_truth'] - o4_posteriors['mode'])

print("Posteriors preprocessed successfully!")
print(f"Sample of computed values:")
print(o4_posteriors[['variable', 'sample_size', 'mean', 'std', 'abs_error_from_mean']].head())

Posteriors preprocessed successfully!
Sample of computed values:
        variable sample_size       mean        std  abs_error_from_mean
11779  Employees          10  34.360099  13.571825            18.565887
11780  Employees          10  41.815436  15.917392            11.110550
11781  Employees          10  40.376314  14.942543            12.549672
11782  Employees          10  37.953738  12.988100            14.972248
11783  Employees          10  42.946544  15.956228             9.979442


In [5]:
# Compute error ratios for posteriors vs baselines
# Matches logic from compare_models.py preprocess_posteriors (lines 359-392)

# Initialize ratio columns
o4_posteriors.loc[:, 'error_ratio_mean'] = np.nan
o4_posteriors.loc[:, 'error_ratio_median'] = np.nan
o4_posteriors.loc[:, 'error_ratio_mode'] = np.nan
o4_posteriors.loc[:, 'std_ratio'] = np.nan

# Compute error ratios by comparing to statistical baselines
# Match by variable, sample_size, trial, and distribution type
for idx, row in o4_posteriors.iterrows():
    # Match baseline by variable, sample_size, trial, and distribution type
    baselines = df[
        (df["approach"].str.contains("statistical", na=False)) &
        (df["sample_size"] == row["sample_size"]) &
        (df["variable"] == row["variable"]) &
        (df["trial"] == row["trial"]) &
        (df["fitted_distribution_type"] == row["fitted_distribution_type"])
    ]
    
    if len(baselines) == 0:
        # This can happen if there's no exact match
        continue
    
    baseline = baselines.iloc[0]
    baseline_error = baseline["abs_error_from_mean"]
    posterior_error = row["abs_error_from_mean"]
    
    if baseline_error > 0:
        o4_posteriors.at[idx, 'error_ratio_mean'] = posterior_error / baseline_error
    if baseline["abs_error_from_median"] > 0:
        o4_posteriors.at[idx, 'error_ratio_median'] = row["abs_error_from_median"] / baseline["abs_error_from_median"]
    if baseline["abs_error_from_mode"] > 0:
        o4_posteriors.at[idx, 'error_ratio_mode'] = row["abs_error_from_mode"] / baseline["abs_error_from_mode"]
    if baseline["std"] > 0:
        o4_posteriors.at[idx, 'std_ratio'] = row["std"] / baseline["std"]

print("Error ratios computed!")
print(f"\nSample of posteriors with error ratios:")
print(o4_posteriors[['variable', 'sample_size', 'trial', 'abs_error_from_mean', 'error_ratio_mean']].head(10))

Error ratios computed!

Sample of posteriors with error ratios:
        variable sample_size  trial  abs_error_from_mean  error_ratio_mean
11779  Employees          10      0            18.565887          0.570977
11780  Employees          10      1            11.110550          0.579240
11781  Employees          10     10            12.549672          0.602907
11782  Employees          10     11            14.972248          0.664041
11783  Employees          10     12             9.979442          0.602370
11784  Employees          10     13            16.871767          0.703847
11785  Employees          10     14            20.109555          0.884544
11786  Employees          10     15             7.482719          0.659234
11787  Employees          10     16            26.020741          0.722764
11788  Employees          10     17             7.937573          0.537995


In [6]:
# Compare o4-mini POSTERIOR vs Statistical Baseline
# Matches logic from compare_models.py compute_error_ratios_and_helped_percentages (lines 428-441)

sample_sizes = ['5', '10', '20', '30']

print("="*80)
print("o4-mini POSTERIOR vs Statistical Baseline Comparison")
print("="*80)
print()

posterior_results = []

for sample_size in sample_sizes:
    # Get posteriors for this sample size
    posteriors_for_n = o4_posteriors[o4_posteriors['sample_size'] == sample_size]
    
    if len(posteriors_for_n) == 0:
        print(f"Sample size {sample_size}: No posterior data")
        continue
    
    # Count how many times posterior is better (error_ratio < 1)
    posterior_better = (posteriors_for_n['error_ratio_mean'] < 1).sum()
    total = len(posteriors_for_n)
    posterior_win_pct = (posterior_better / total * 100) if total > 0 else 0
    
    # Get mean error ratio
    mean_error_ratio = posteriors_for_n['error_ratio_mean'].mean()
    
    print(f"\nSample Size N={sample_size}:")
    print(f"  Posterior better: {posterior_better}/{total} ({posterior_win_pct:.1f}%)")
    print(f"  Mean error ratio: {mean_error_ratio:.3f}")
    print(f"  (error_ratio < 1 means posterior has lower error than baseline)")
    
    posterior_results.append({
        'sample_size': sample_size,
        'posterior_better_count': posterior_better,
        'total': total,
        'win_percentage': posterior_win_pct,
        'mean_error_ratio': mean_error_ratio
    })

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
for result in posterior_results:
    print(f"N={result['sample_size']}: {result['win_percentage']:.1f}% win rate (error ratio: {result['mean_error_ratio']:.3f})")

o4-mini POSTERIOR vs Statistical Baseline Comparison


Sample Size N=5:
  Posterior better: 1061/1525 (69.6%)
  Mean error ratio: 4.828
  (error_ratio < 1 means posterior has lower error than baseline)

Sample Size N=10:
  Posterior better: 1166/1525 (76.5%)
  Mean error ratio: 1.738
  (error_ratio < 1 means posterior has lower error than baseline)

Sample Size N=20:
  Posterior better: 1221/1525 (80.1%)
  Mean error ratio: 1.907
  (error_ratio < 1 means posterior has lower error than baseline)

Sample Size N=30:
  Posterior better: 1245/1525 (81.6%)
  Mean error ratio: 1.450
  (error_ratio < 1 means posterior has lower error than baseline)

SUMMARY
N=5: 69.6% win rate (error ratio: 4.828)
N=10: 76.5% win rate (error ratio: 1.738)
N=20: 80.1% win rate (error ratio: 1.907)
N=30: 81.6% win rate (error ratio: 1.450)


In [7]:
# Deep dive: Why does mean error ratio increase from N=10 to N=20?
# Let's look at the distribution of error ratios and identify outliers

sample_sizes = ['5', '10', '20', '30']

print("="*80)
print("Error Ratio Distribution Analysis")
print("="*80)
print()

for sample_size in sample_sizes:
    posteriors_for_n = o4_posteriors[o4_posteriors['sample_size'] == sample_size]
    
    if len(posteriors_for_n) == 0:
        continue
    
    error_ratios = posteriors_for_n['error_ratio_mean'].dropna()
    
    print(f"\nSample Size N={sample_size}:")
    print(f"  Count: {len(error_ratios)}")
    print(f"  Mean: {error_ratios.mean():.3f}")
    print(f"  Median: {error_ratios.median():.3f}")
    print(f"  Std: {error_ratios.std():.3f}")
    print(f"  Min: {error_ratios.min():.3f}")
    print(f"  Max: {error_ratios.max():.3f}")
    print(f"  25th percentile: {error_ratios.quantile(0.25):.3f}")
    print(f"  75th percentile: {error_ratios.quantile(0.75):.3f}")
    print(f"  95th percentile: {error_ratios.quantile(0.95):.3f}")
    print(f"  99th percentile: {error_ratios.quantile(0.99):.3f}")
    
    # Count extreme outliers
    extreme_outliers = (error_ratios > 10).sum()
    very_extreme = (error_ratios > 100).sum()
    print(f"  Outliers > 10: {extreme_outliers} ({extreme_outliers/len(error_ratios)*100:.1f}%)")
    print(f"  Outliers > 100: {very_extreme} ({very_extreme/len(error_ratios)*100:.1f}%)")

Error Ratio Distribution Analysis


Sample Size N=5:
  Count: 1525
  Mean: 4.828
  Median: 0.868
  Std: 109.569
  Min: 0.002
  Max: 4274.440
  25th percentile: 0.628
  75th percentile: 1.091
  95th percentile: 5.331
  99th percentile: 35.328
  Outliers > 10: 45 (3.0%)
  Outliers > 100: 1 (0.1%)

Sample Size N=10:
  Count: 1525
  Mean: 1.738
  Median: 0.867
  Std: 5.323
  Min: 0.013
  Max: 143.797
  25th percentile: 0.682
  75th percentile: 0.985
  95th percentile: 4.767
  99th percentile: 24.298
  Outliers > 10: 43 (2.8%)
  Outliers > 100: 1 (0.1%)

Sample Size N=20:
  Count: 1523
  Mean: 1.907
  Median: 0.904
  Std: 6.825
  Min: 0.006
  Max: 75.543
  25th percentile: 0.797
  75th percentile: 0.976
  95th percentile: 3.689
  99th percentile: 35.770
  Outliers > 10: 29 (1.9%)
  Outliers > 100: 0 (0.0%)

Sample Size N=30:
  Count: 1525
  Mean: 1.450
  Median: 0.930
  Std: 2.772
  Min: 0.011
  Max: 27.759
  25th percentile: 0.852
  75th percentile: 0.978
  95th percentile: 3.894
  99th pe

In [8]:
# Investigate outliers: which variables and why?
# Compare N=10 vs N=20 specifically

print("="*80)
print("Outlier Investigation: N=10 vs N=20")
print("="*80)
print()

for sample_size in ['10', '20']:
    posteriors_for_n = o4_posteriors[o4_posteriors['sample_size'] == sample_size]
    error_ratios = posteriors_for_n['error_ratio_mean'].dropna()
    
    # Find the top 10 worst outliers
    worst_cases = posteriors_for_n.nlargest(10, 'error_ratio_mean')
    
    print(f"\nSample Size N={sample_size} - Top 10 Worst Error Ratios:")
    print("-" * 80)
    for idx, row in worst_cases.iterrows():
        print(f"  Variable: {row['variable']}, Trial: {row['trial']}")
        print(f"    Error ratio: {row['error_ratio_mean']:.1f}")
        print(f"    Posterior error: {row['abs_error_from_mean']:.4f}")
        # Find the corresponding baseline
        baselines = df[
            (df["approach"].str.contains("statistical", na=False)) &
            (df["sample_size"] == row["sample_size"]) &
            (df["variable"] == row["variable"]) &
            (df["trial"] == row["trial"]) &
            (df["fitted_distribution_type"] == row["fitted_distribution_type"])
        ]
        if len(baselines) > 0:
            baseline_error = baselines.iloc[0]["abs_error_from_mean"]
            print(f"    Baseline error: {baseline_error:.4f}")
        print()

Outlier Investigation: N=10 vs N=20


Sample Size N=10 - Top 10 Worst Error Ratios:
--------------------------------------------------------------------------------
  Variable: medium_0, Trial: 2
    Error ratio: 143.8
    Posterior error: 4.6585
    Baseline error: 0.0324

  Variable: easy_0, Trial: 16
    Error ratio: 68.5
    Posterior error: 10.2681
    Baseline error: 0.1500

  Variable: hard_9, Trial: 18
    Error ratio: 54.8
    Posterior error: 0.3880
    Baseline error: 0.0071

  Variable: hard_5, Trial: 23
    Error ratio: 26.1
    Posterior error: 10.9060
    Baseline error: 0.4186

  Variable: easy_7, Trial: 1
    Error ratio: 25.7
    Posterior error: 0.3121
    Baseline error: 0.0121

  Variable: easy_7, Trial: 14
    Error ratio: 25.7
    Posterior error: 0.3121
    Baseline error: 0.0121

  Variable: easy_7, Trial: 16
    Error ratio: 25.7
    Posterior error: 0.3121
    Baseline error: 0.0121

  Variable: easy_7, Trial: 21
    Error ratio: 25.7
    Posterior error: 0.3

In [9]:
# Hypothesis: Larger sample sizes produce SMALLER baseline errors
# When baseline error is very small, even a small posterior error creates a large ratio
# This is the "division by small number" problem

print("="*80)
print("Baseline Error Magnitude Analysis")
print("="*80)
print()

for sample_size in ['5', '10', '20', '30']:
    # Get all statistical baselines for this sample size
    baselines_for_n = df[
        (df["approach"].str.contains("statistical", na=False)) &
        (df["sample_size"] == sample_size)
    ]
    
    if len(baselines_for_n) == 0:
        continue
    
    baseline_errors = baselines_for_n['abs_error_from_mean'].dropna()
    
    print(f"\nSample Size N={sample_size}:")
    print(f"  Baseline error mean: {baseline_errors.mean():.4f}")
    print(f"  Baseline error median: {baseline_errors.median():.4f}")
    print(f"  Baseline error std: {baseline_errors.std():.4f}")
    print(f"  Baseline errors < 0.01: {(baseline_errors < 0.01).sum()} ({(baseline_errors < 0.01).sum()/len(baseline_errors)*100:.1f}%)")
    print(f"  Baseline errors < 0.001: {(baseline_errors < 0.001).sum()} ({(baseline_errors < 0.001).sum()/len(baseline_errors)*100:.1f}%)")

print("\n" + "="*80)
print("Key Insight:")
print("If larger N → smaller baseline errors → division by small numbers → inflated ratios")
print("="*80)

Baseline Error Magnitude Analysis


Sample Size N=5:
  Baseline error mean: 31.6740
  Baseline error median: 11.3276
  Baseline error std: 257.7252
  Baseline errors < 0.01: 25 (0.9%)
  Baseline errors < 0.001: 0 (0.0%)

Sample Size N=10:
  Baseline error mean: 27.3442
  Baseline error median: 10.4362
  Baseline error std: 63.2004
  Baseline errors < 0.01: 17 (0.6%)
  Baseline errors < 0.001: 0 (0.0%)

Sample Size N=20:
  Baseline error mean: 23.9844
  Baseline error median: 10.0397
  Baseline error std: 73.5425
  Baseline errors < 0.01: 37 (1.4%)
  Baseline errors < 0.001: 2 (0.1%)

Sample Size N=30:
  Baseline error mean: 24.9978
  Baseline error median: 9.2321
  Baseline error std: 73.5114
  Baseline errors < 0.01: 26 (1.0%)
  Baseline errors < 0.001: 0 (0.0%)

Key Insight:
If larger N → smaller baseline errors → division by small numbers → inflated ratios


In [ ]:
# Paradox Investigation: Why does win rate INCREASE while median ratio also increases?
# 
# Median ratio increasing (0.868 → 0.930) means posterior is getting relatively WORSE
# But win rate increasing (69.6% → 81.6%) means posterior is winning MORE often
#
# This can only happen if the DISTRIBUTION is changing shape

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
sample_sizes = ['5', '10', '20', '30']

for i, sample_size in enumerate(sample_sizes):
    ax = axes[i // 2, i % 2]
    posteriors_for_n = o4_posteriors[o4_posteriors['sample_size'] == sample_size]
    error_ratios = posteriors_for_n['error_ratio_mean'].dropna()
    
    # Clip for visualization (ignore extreme outliers)
    error_ratios_clipped = error_ratios.clip(upper=5)
    
    ax.hist(error_ratios_clipped, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(x=1.0, color='red', linestyle='--', label='Ratio = 1 (tie)')
    ax.axvline(x=error_ratios.median(), color='green', linestyle='-', label=f'Median = {error_ratios.median():.3f}')
    
    win_pct = (error_ratios < 1).sum() / len(error_ratios) * 100
    ax.set_title(f'N={sample_size}: Win={win_pct:.1f}%, Median={error_ratios.median():.3f}')
    ax.set_xlabel('Error Ratio (clipped at 5)')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig('error_ratio_distributions.png', dpi=150)
plt.show()

print("\nDetailed distribution analysis:")
for sample_size in sample_sizes:
    posteriors_for_n = o4_posteriors[o4_posteriors['sample_size'] == sample_size]
    error_ratios = posteriors_for_n['error_ratio_mean'].dropna()
    
    # Check distribution around the threshold
    below_0_5 = (error_ratios < 0.5).sum() / len(error_ratios) * 100
    below_0_8 = (error_ratios < 0.8).sum() / len(error_ratios) * 100
    below_1_0 = (error_ratios < 1.0).sum() / len(error_ratios) * 100
    below_1_2 = (error_ratios < 1.2).sum() / len(error_ratios) * 100
    
    print(f"\nN={sample_size}:")
    print(f"  < 0.5: {below_0_5:.1f}%")
    print(f"  < 0.8: {below_0_8:.1f}%")
    print(f"  < 1.0 (wins): {below_1_0:.1f}%")
    print(f"  < 1.2: {below_1_2:.1f}%")